# LLM Post-Training on Google Colab

This notebook provides a complete template for training language models using various post-training techniques on Google Colab.

**What's Included:**
- SFT (Supervised Fine-Tuning) for text and multimodal models
- DPO (Direct Preference Optimization) for preference learning
- PPO/RLHF for reinforcement learning from human feedback

**Hardware Requirements:**
- Free Tier: CLIP, GPT-2, small models
- Pro ($10/mo): LLaVA-7B with LoRA
- Pro+ ($50/mo): LLaMA-13B with LoRA

**Runtime Setup:**
1. Runtime → Change runtime type
2. Hardware accelerator: GPU (T4, V100, or A100)
3. Click Save

---

📚 **Documentation:** See `docs/google_colab_guide.md` in the repository

## Table of Contents

1. [Setup](#setup)
2. [GPU Check](#gpu-check)
3. [Storage Setup](#storage)
4. [Training Examples](#training)
   - [CLIP Image-Text Training](#clip)
   - [LLaVA Vision-Language Training](#llava)
   - [GPT-2 Text Training](#gpt2)
   - [DPO Preference Learning](#dpo)
5. [Evaluation](#evaluation)
6. [Download Models](#download)
7. [Troubleshooting](#troubleshooting)

## 1. Setup

Install dependencies and clone the repository.

In [ ]:
# Clone repository (replace with your fork/repo)
!git clone https://github.com/your-username/llm-post-training.git
%cd llm-post-training

# Check we're in the right directory
!ls -la

In [ ]:
# Install dependencies (this takes 2-3 minutes)
!pip install -q -r requirements/base.txt
!pip install -q -r requirements/multimodal.txt

print("✓ Installation complete!")

## 2. GPU Check

Verify GPU is available and check which GPU you got.

In [ ]:
# Check GPU with nvidia-smi
!nvidia-smi

In [ ]:
# Check PyTorch can see GPU
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime → Change runtime type → Select GPU")

## 3. Storage Setup

Mount Google Drive to save outputs permanently.

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create output directory in Drive
!mkdir -p /content/drive/MyDrive/llm_outputs

print("✓ Google Drive mounted at /content/drive/")
print("✓ Outputs will be saved to: /content/drive/MyDrive/llm_outputs")

## 4. Training Examples

Choose a training example based on your model and task.

### 4.1 CLIP Image-Text Training

Train CLIP on image-caption pairs.

**Memory:** ~3 GB VRAM  
**Time:** ~2-3 minutes for 1000 samples on T4  
**LoRA:** Not needed (CLIP is small, and LoRA is broken for CLIP)

In [ ]:
# Train CLIP on synthetic data (quick test)
!python scripts/train/train_multimodal.py \
    experiment=clip_image_caption \
    data.dataset_name=synthetic \
    data.max_train_samples=1000 \
    training.output_dir=/content/drive/MyDrive/llm_outputs/clip_synthetic \
    training.num_epochs=3 \
    training.per_device_train_batch_size=32 \
    training.fp16=true \
    training.logging_steps=10

In [ ]:
# Train CLIP on COCO (real data - takes longer)
!python scripts/train/train_multimodal.py \
    experiment=clip_image_caption \
    data.dataset_name=coco \
    data.max_train_samples=5000 \
    training.output_dir=/content/drive/MyDrive/llm_outputs/clip_coco \
    training.num_epochs=3 \
    training.per_device_train_batch_size=32 \
    training.fp16=true

### 4.2 LLaVA Vision-Language Training

Train LLaVA (7B) with LoRA and 4-bit quantization.

**Memory:** ~6 GB VRAM (with 4-bit + LoRA)  
**Time:** ~30-40 minutes for 1000 samples on T4  
**Requirements:** Colab Pro recommended (Free tier works but may timeout)

In [ ]:
# Train LLaVA-7B with LoRA (4-bit quantization essential)
!python scripts/train/train_multimodal.py \
    experiment=llava_instruction \
    model.use_lora=true \
    model.use_4bit=true \
    data.max_train_samples=1000 \
    training.output_dir=/content/drive/MyDrive/llm_outputs/llava_instruction \
    training.num_epochs=1 \
    training.per_device_train_batch_size=4 \
    training.gradient_accumulation_steps=4 \
    training.fp16=true \
    training.logging_steps=10 \
    training.save_steps=100

### 4.3 GPT-2 Text Training

Train GPT-2 on conversational data.

**Memory:** ~2 GB VRAM (with LoRA)  
**Time:** ~5-10 minutes for 5000 samples on T4

In [ ]:
# Train GPT-2 with LoRA
!python scripts/train/train_sft.py \
    experiment=gpt2_conversation \
    model.use_lora=true \
    data.max_train_samples=5000 \
    training.output_dir=/content/drive/MyDrive/llm_outputs/gpt2_conversation \
    training.num_epochs=3 \
    training.per_device_train_batch_size=16 \
    training.fp16=true

### 4.4 DPO Preference Learning

Train using Direct Preference Optimization on preference pairs.

**Memory:** ~3 GB VRAM (with LoRA)  
**Time:** ~10-15 minutes for 2000 pairs on T4

In [ ]:
# Train GPT-2 with DPO
!python scripts/train/train_dpo.py \
    experiment=gpt2_dpo \
    model.use_lora=true \
    data.dataset_name=anthropic_hh \
    data.max_train_samples=2000 \
    training.output_dir=/content/drive/MyDrive/llm_outputs/gpt2_dpo \
    training.num_epochs=1 \
    training.per_device_train_batch_size=8 \
    training.beta=0.1 \
    training.fp16=true

## 5. Evaluation

Evaluate trained models on test data.

In [ ]:
# Evaluate CLIP model
!python scripts/evaluate/evaluate_model.py \
    model_path=/content/drive/MyDrive/llm_outputs/clip_synthetic \
    model_type=clip \
    eval_dataset=coco \
    metrics=clip_score

In [ ]:
# Evaluate text model (perplexity, BLEU, ROUGE)
!python scripts/evaluate/evaluate_model.py \
    model_path=/content/drive/MyDrive/llm_outputs/gpt2_conversation \
    model_type=gpt2 \
    eval_dataset=conversation_test \
    metrics=perplexity,bleu,rouge

## 6. Download Models

Download trained models to your local machine.

In [ ]:
# Zip model checkpoint
!zip -r /content/model_checkpoint.zip /content/drive/MyDrive/llm_outputs/clip_synthetic

# Download (triggers browser download)
from google.colab import files
files.download('/content/model_checkpoint.zip')

### Upload to HuggingFace Hub (Optional)

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
login()  # Enter your token when prompted

# Upload model
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="/content/drive/MyDrive/llm_outputs/clip_synthetic",
    repo_id="your-username/clip-finetuned",
    repo_type="model"
)

print("✓ Model uploaded to HuggingFace Hub!")

## 7. Troubleshooting

### Out of Memory (OOM)

**Solutions:**
1. Reduce batch size: `training.per_device_train_batch_size=4`
2. Increase gradient accumulation: `training.gradient_accumulation_steps=4`
3. Enable 4-bit quantization: `model.use_4bit=true`
4. Use gradient checkpointing: `model.gradient_checkpointing=true`

### Session Disconnected

**Prevention:**
- Save to Google Drive (already configured above)
- Use frequent checkpoints: `training.save_steps=100`
- Upgrade to Colab Pro for background execution

**Recovery:**
```bash
# Resume from checkpoint
!python scripts/train/train_sft.py \
    experiment=gpt2_conversation \
    training.resume_from_checkpoint=/content/drive/MyDrive/llm_outputs/gpt2_conversation/checkpoint-500
```

### Disk Space Full

**Clear cache:**
```bash
!rm -rf ~/.cache/huggingface/hub
!rm -rf /content/llm-post-training/outputs
```

### Package Version Conflicts

**Restart runtime:**
Runtime → Restart runtime (then re-run setup cells)

---

## Additional Resources

- **Full Documentation:** `docs/google_colab_guide.md`
- **Known Issues:** `docs/known_issues.md`
- **Configuration Guide:** `docs/CONFIGURATION_GUIDE.md`
- **Model Selection:** `docs/model_selection_guide.md`

## Support

- Issues: https://github.com/your-username/llm-post-training/issues
- Discussions: https://github.com/your-username/llm-post-training/discussions

## Cleanup (Optional)

Free up disk space when done.

In [ ]:
# Clear PyTorch cache
import torch
torch.cuda.empty_cache()

# Clear HuggingFace cache
!rm -rf ~/.cache/huggingface/hub

# Remove local outputs (if saved to Drive)
!rm -rf /content/llm-post-training/outputs

print("✓ Cleanup complete!")